In [ ]:
# Import required libraries
import pandas as pd
import json
import regex as re

In [ ]:
# Conversion sheets for transliterations if necessary
conversions = pd.read_excel('../data/conversions.xlsx')
conversions

In [ ]:
# Get data straight from the current working file
file = open("../_data/PhyrexianAPI.json", encoding='utf-8')
phy_all = json.loads(file.read())

In [ ]:
# All words from dictionary
words = pd.DataFrame.from_dict(phy_all["dictionary"], orient="index")
words

In [ ]:
# All asmples from dictionary
raw = pd.DataFrame.from_dict(phy_all["samples"], orient="index")
raw

In [ ]:
def parse_translit(raw:str):
    # For transliteration, separate out words in a string of text
    parsed = [char for char in raw]
    char_i = 0
    while char_i < len(parsed):
        if parsed[char_i] in ["-", "+"]:
            parsed[char_i] = parsed[char_i] + parsed[char_i+1]
            del parsed[char_i+1]
            char_i -= 1
        if parsed[char_i] in ["e", "w", "v", "r", "d", "q", "x", "y"]:
            parsed[char_i-1] = parsed[char_i-1] + parsed[char_i]
            del parsed[char_i]
            char_i -= 1
        char_i += 1
    return parsed

def translit_to_font(letters:list, conversion_sheet:dict):
    # Convert transliterated text to a particular font
    result = ''
    for char in letters:
        if char in conversion_sheet.keys():
            result = result + conversion_sheet[char]
        else:
            result = result + char
    return result

In [ ]:
# Useful variables
vowels = 'aeiouyøɒə'
vowels_re = '[aeiouyøɒə]*'
roots = words[words['type'].isin(['root', 'name', 'unclear'])]['phyrexian'].tolist()
composites = words.query('type == "composite"')['phyrexian'].to_list()
moods = words.query('type == "marker"')['phyrexian'].to_list()

In [ ]:
def unvowel(query):
    # Inputs phyrexian text, outputs same text without vowels
    new = query
    for vow in vowels:
        new = new.replace(vow, '')
    return new
    
def raw_split(raw_sample):
    # Get all words from sample in phyrexian
    return re.split('\.\^|\-|\ |\:', raw_sample.rstrip('.').lstrip('^'))

def unroot_raw(raw_thing):
    # Get roots for every word in raw sample in phyrexian
    result = []
    raw_split = re.split('\.\^|\-|\ |\:', raw_thing)
    for word in raw_split:
        un_word = unvowel(word)
        matches = []
        for root in roots:
            un_root = unvowel(root)
            if un_root in un_word:
                matches.append(un_root)
        stripped = word
        for match in matches:
            if stripped != '':
                stripped = stripped.replace(re.search(vowels_re.join(match), stripped).string, '')
        if stripped != '':
            result.append(stripped)
    return result

In [ ]:
# Go through samples and get uses of words
all_matches = []
for idx in words.index: # For each word
    matches = []

    for raw_idx in raw.index: # Go through each sample
        name = raw.loc[raw_idx, 'Name']
        raw_listed = raw_split(raw.loc[raw_idx, 'Regular'])
        for word in raw_listed: # For each word in that sample
            match words.loc[idx, 'type']:
                case 'root' | 'composite' | 'unclear': # If current dictionary word is a "normal" word, accept unvoweling
                    if re.search(vowels_re.join(words.loc[idx, 'phyrexian']), word):
                        if name not in matches:
                            matches.append(name)
                case 'name' | 'number': # Otherwise take word as is
                    if words.loc[idx, 'phyrexian'] in word:
                        if name not in matches:
                            matches.append(name)
                case 'marker': # Not implemented
                    if words.loc[idx, 'phyrexian'] in word:
                        if word.replace(words.loc[idx, 'phyrexian'], '') in moods + ['']: # Makes sure the matched word ONLY has that mood
                            if name not in matches:
                                matches.append(name)
    all_matches.append(matches)

In [ ]:
all_matches

In [ ]:
# Assign examples
words['examples'] = all_matches

In [ ]:
# Build final dict
final = {}
final['dictionary'] = words.to_dict(orient='index')
final['samples'] = raw.to_dict(orient='index')
final_json = json.dumps(final, indent = 4, ensure_ascii=False).replace('NaN', 'null')

In [ ]:
# Write to provisory file
out = open('../data/PhyrexianAPI', 'w', encoding='utf-8')
out.write(final_json)
out.close()

In [ ]:
def reposition(index_from:int, index_to:int):
    # Changes the location of a single entry on the dictionary
    working_index = words.index.to_list()
    working_index.insert(index_to, str(index_from))
    working_index.pop(index_from + 1)
    repositioned_df = words.reindex(working_index)
    repositioned_df.index = range(len(repositioned_df.index))
    return repositioned